In [1]:
import pandas as pd

In [2]:
# 1. Load the data safely
df = pd.read_csv('RetailCleaned.csv', low_memory=False)

# 2. Force the values into the correct data alignment by dropping rows corrupted by text shifts
df = df[df['Quantity'].astype(str).str.replace('-', '', 1).str.isnumeric()]

In [3]:

# 3. Clean and convert to match Mansi Mam's exact Dtypes from the screenshot
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='mixed', errors='coerce')
df['Quantity'] = df['Quantity'].astype('int64')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce').astype('float64')
df['Customer_ID'] = pd.to_numeric(df['Customer_ID'], errors='coerce').astype('float64')

# Ensure the text fields are mapped explicitly as string objects
df['StockCode'] = df['StockCode'].astype(str)
df['Description'] = df['Description'].astype(str)
df['Country'] = df['Country'].astype(str)
df['SourceSheet'] = df['SourceSheet'].astype(str)
df['Invoice'] = df['Invoice'].astype(object)

# 4. Run the check to see the exact matching matrix!
df.info()


<class 'pandas.DataFrame'>
Index: 1033025 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033025 non-null  object        
 1   StockCode    1033025 non-null  str           
 2   Description  1033025 non-null  str           
 3   Quantity     1033025 non-null  int64         
 4   InvoiceDate  1033025 non-null  datetime64[us]
 5   Price        1033025 non-null  float64       
 6   Customer_ID  797876 non-null   float64       
 7   Country      1033025 non-null  str           
 8   SourceSheet  1033025 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(4)
memory usage: 133.6+ MB


In [4]:
df.shape

(1033025, 9)

In [5]:
# Check the count of missing items to see if we match her null summaries
print("--- Missing Values Tracker ---")
print(df.isnull().sum())


--- Missing Values Tracker ---
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer_ID    235149
Country             0
SourceSheet         0
dtype: int64


In [6]:
# calculating recency, frequency, and monetary 
rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity']> 0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice', 'nunique'),
    Monetary = ('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()

In [7]:
rfm.head()

,Customer_ID,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


In [8]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels = [5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5 , labels = [1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels = [1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3 :
        return 'Loyal Customers'
    elif row['R_Score'] <=2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'

    else:
        return 'Lost'

rfm['segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90

In [9]:
rfm.head()

,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,segment,IsChurned
0,12346.0,325,12,77556.46,2,5,5,At Risk,True
1,12347.0,2,8,4921.53,5,4,5,Champions,False
2,12348.0,75,5,2019.40,3,4,4,Loyal Customers,False
3,12349.0,18,4,4428.69,5,3,5,Loyal Customers,False
4,12350.0,310,1,334.40,2,1,2,Lost,True


In [15]:
df = df[(df['Customer_ID'].notna()) & (df['Quantity']> 0)].copy()
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[['Customer_ID','Recency','Frequency','Monetary','R_Score','F_Score','M_Score','segment','IsChurned']],
    on='Customer_ID',how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)